In [1]:
!wget -q --show-progress \
  "https://zenodo.org/records/8196385/files/HDFS_v1.zip?download=1" \
  -O /content/HDFS_v1.zip

/content/HDFS_v1.zi 100%[===================>] 178.00M  2.71MB/s    in 88s     


In [2]:
!unzip -q /content/HDFS_v1.zip -d /content/HDFS_dataset

In [3]:
from pathlib import Path

dataset_dir = Path("/content/HDFS_dataset")

for file in dataset_dir.rglob("*"):
    if file.is_file() and (
        file.name == "HDFS.log"
        or file.name == "anomaly_label.csv"
    ):
        print(file, f"({file.stat().st_size / 1024**2:.2f} MB)")

/content/HDFS_dataset/HDFS.log (1504.88 MB)
/content/HDFS_dataset/preprocessed/anomaly_label.csv (17.77 MB)


In [4]:
from pathlib import Path
import pandas as pd

RAW_LOG = Path("/content/HDFS_dataset/HDFS.log")
LABELS = Path("/content/HDFS_dataset/preprocessed/anomaly_label.csv")

# 1. Inspect raw log format
print("=== FIRST 5 RAW LOG ENTRIES ===")
with RAW_LOG.open("r", encoding="utf-8", errors="replace") as f:
    for _ in range(5):
        print(next(f).rstrip())

# 2. Inspect label structure
labels = pd.read_csv(LABELS)

print("\n=== LABEL DATASET ===")
print("Shape:", labels.shape)
print("\nColumns:", labels.columns.tolist())
print("\nFirst 5 rows:")
print(labels.head())

# 3. Inspect label distribution
print("\n=== LABEL DISTRIBUTION ===")
print(labels["Label"].value_counts(dropna=False))

# 4. Check missing values and duplicates
print("\n=== DATA QUALITY ===")
print("Missing values:")
print(labels.isnull().sum())

print("\nDuplicate BlockIds:", labels["BlockId"].duplicated().sum())

=== FIRST 5 RAW LOG ENTRIES ===
081109 203518 143 INFO dfs.DataNode$DataXceiver: Receiving block blk_-1608999687919862906 src: /10.250.19.102:54106 dest: /10.250.19.102:50010
081109 203518 35 INFO dfs.FSNamesystem: BLOCK* NameSystem.allocateBlock: /mnt/hadoop/mapred/system/job_200811092030_0001/job.jar. blk_-1608999687919862906
081109 203519 143 INFO dfs.DataNode$DataXceiver: Receiving block blk_-1608999687919862906 src: /10.250.10.6:40524 dest: /10.250.10.6:50010
081109 203519 145 INFO dfs.DataNode$DataXceiver: Receiving block blk_-1608999687919862906 src: /10.250.14.224:42420 dest: /10.250.14.224:50010
081109 203519 145 INFO dfs.DataNode$PacketResponder: PacketResponder 1 for block blk_-1608999687919862906 terminating

=== LABEL DATASET ===
Shape: (575061, 2)

Columns: ['BlockId', 'Label']

First 5 rows:
                    BlockId    Label
0  blk_-1608999687919862906   Normal
1   blk_7503483334202473044   Normal
2  blk_-3544583377289625738  Anomaly
3  blk_-9073992586687739851   Norm

In [5]:
import re
from collections import Counter
from datetime import datetime

block_pattern = re.compile(r"blk_-?\d+")
timestamp_pattern = re.compile(r"^(\d{6})\s+(\d{6})\s+")

total_lines = 0
lines_without_blocks = 0
lines_with_multiple_blocks = 0
invalid_timestamps = 0
timestamp_regressions = 0

unique_blocks = set()
levels = Counter()

first_timestamp = None
last_timestamp = None
min_timestamp = None
max_timestamp = None
previous_timestamp = None

with open(RAW_LOG, "r", encoding="utf-8", errors="replace") as f:
    for line in f:
        total_lines += 1

        # Extract block identifiers
        blocks = set(block_pattern.findall(line))
        unique_blocks.update(blocks)

        if not blocks:
            lines_without_blocks += 1
        elif len(blocks) > 1:
            lines_with_multiple_blocks += 1

        # Validate timestamp
        match = timestamp_pattern.match(line)

        if match:
            try:
                ts = datetime.strptime(
                    match.group(1) + match.group(2),
                    "%y%m%d%H%M%S"
                )

                if first_timestamp is None:
                    first_timestamp = ts

                if previous_timestamp and ts < previous_timestamp:
                    timestamp_regressions += 1

                min_timestamp = (
                    ts if min_timestamp is None
                    else min(min_timestamp, ts)
                )
                max_timestamp = (
                    ts if max_timestamp is None
                    else max(max_timestamp, ts)
                )

                previous_timestamp = ts
                last_timestamp = ts

            except ValueError:
                invalid_timestamps += 1
        else:
            invalid_timestamps += 1

        # Extract log level
        parts = line.split(maxsplit=4)
        if len(parts) >= 4:
            levels[parts[3]] += 1

labeled_blocks = set(labels["BlockId"])

print("=== RAW LOG AUDIT ===")
print("Total log lines:", total_lines)
print("Unique block IDs:", len(unique_blocks))
print("Lines without block IDs:", lines_without_blocks)
print("Lines with multiple distinct block IDs:", lines_with_multiple_blocks)

print("\n=== TIMESTAMP AUDIT ===")
print("First timestamp:", first_timestamp)
print("Last timestamp:", last_timestamp)
print("Minimum timestamp:", min_timestamp)
print("Maximum timestamp:", max_timestamp)
print("Invalid timestamps:", invalid_timestamps)
print("Timestamp regressions:", timestamp_regressions)

print("\n=== LOG LEVELS ===")
print(levels)

print("\n=== BLOCK COVERAGE ===")
print("Labeled blocks:", len(labeled_blocks))
print("Labeled blocks missing from logs:", len(labeled_blocks - unique_blocks))
print("Unlabeled blocks found in logs:", len(unique_blocks - labeled_blocks))

=== RAW LOG AUDIT ===
Total log lines: 11175629
Unique block IDs: 575061
Lines without block IDs: 0
Lines with multiple distinct block IDs: 0

=== TIMESTAMP AUDIT ===
First timestamp: 2008-11-09 20:35:18
Last timestamp: 2008-11-11 11:16:28
Minimum timestamp: 2008-11-09 20:35:18
Maximum timestamp: 2008-11-11 11:16:28
Invalid timestamps: 0
Timestamp regressions: 0

=== LOG LEVELS ===
Counter({'INFO': 10812836, 'WARN': 362793})

=== BLOCK COVERAGE ===
Labeled blocks: 575061
Labeled blocks missing from logs: 0
Unlabeled blocks found in logs: 0


In [6]:
from collections import defaultdict

examples = defaultdict(list)

with open(RAW_LOG, "r", encoding="utf-8", errors="replace") as f:
    for line in f:
        parts = line.strip().split(maxsplit=5)

        if len(parts) < 6:
            continue

        component = parts[4]

        if len(examples[component]) < 3:
            examples[component].append(parts[5])

        if len(examples) >= 10 and all(
            len(v) >= 3 for v in examples.values()
        ):
            break

for component, messages in examples.items():
    print(f"\n=== {component} ===")
    for message in messages:
        print("-", message)


=== dfs.DataNode$DataXceiver: ===
- Receiving block blk_-1608999687919862906 src: /10.250.19.102:54106 dest: /10.250.19.102:50010
- Receiving block blk_-1608999687919862906 src: /10.250.10.6:40524 dest: /10.250.10.6:50010
- Receiving block blk_-1608999687919862906 src: /10.250.14.224:42420 dest: /10.250.14.224:50010

=== dfs.FSNamesystem: ===
- BLOCK* NameSystem.allocateBlock: /mnt/hadoop/mapred/system/job_200811092030_0001/job.jar. blk_-1608999687919862906
- BLOCK* NameSystem.addStoredBlock: blockMap updated: 10.250.10.6:50010 is added to blk_-1608999687919862906 size 91178
- BLOCK* NameSystem.addStoredBlock: blockMap updated: 10.251.111.209:50010 is added to blk_-1608999687919862906 size 91178

=== dfs.DataNode$PacketResponder: ===
- PacketResponder 1 for block blk_-1608999687919862906 terminating
- PacketResponder 2 for block blk_-1608999687919862906 terminating
- Received block blk_-1608999687919862906 of size 91178 from /10.250.10.6

=== dfs.DataNode$DataTransfer: ===
- 10.250.14

In [7]:
import re
import pandas as pd
from collections import defaultdict

block_pattern = re.compile(r"blk_-?\d+")

block_info = {}

with open(RAW_LOG, "r", encoding="utf-8", errors="replace") as f:
    for line_number, line in enumerate(f):

        # YYYY is 2008 for this dataset.
        # Format in raw logs: YYMMDD HHMMSS
        parts = line.split(maxsplit=2)

        timestamp = "20" + parts[0] + parts[1]

        block_id = block_pattern.search(line).group()

        if block_id not in block_info:
            block_info[block_id] = {
                "BlockId": block_id,
                "first_timestamp": timestamp,
                "last_timestamp": timestamp,
                "first_line": line_number,
                "event_count": 0
            }

        block_info[block_id]["last_timestamp"] = timestamp
        block_info[block_id]["event_count"] += 1

metadata = pd.DataFrame(block_info.values())

# Attach ground-truth labels
metadata = metadata.merge(
    labels,
    on="BlockId",
    how="left",
    validate="one_to_one"
)

assert len(metadata) == len(labels)
assert metadata["Label"].notna().all()

# Sort by first observation time.
# Original line number breaks timestamp ties.
metadata = metadata.sort_values(
    ["first_timestamp", "first_line"]
).reset_index(drop=True)

n = len(metadata)

train_end = int(n * 0.70)
val_end = int(n * 0.85)

metadata["split"] = "test"
metadata.loc[:train_end - 1, "split"] = "train"
metadata.loc[train_end:val_end - 1, "split"] = "validation"

print("=== PROVISIONAL CHRONOLOGICAL SPLIT ===")
print(pd.crosstab(metadata["split"], metadata["Label"]))

print("\n=== EVENT COUNT STATISTICS ===")
print(metadata["event_count"].describe())

print("\n=== SPLIT BOUNDARIES ===")
for split in ["train", "validation", "test"]:
    subset = metadata[metadata["split"] == split]
    print(
        split,
        "| blocks:", len(subset),
        "| first:", subset["first_timestamp"].min(),
        "| last:", subset["last_timestamp"].max()
    )

# A block may start in one period and finish in another.
train_boundary = metadata.loc[train_end, "first_timestamp"]
val_boundary = metadata.loc[val_end, "first_timestamp"]

train_crossing = (
    (metadata["first_timestamp"] < train_boundary) &
    (metadata["last_timestamp"] >= train_boundary)
).sum()

val_crossing = (
    (metadata["first_timestamp"] < val_boundary) &
    (metadata["last_timestamp"] >= val_boundary)
).sum()

print("\n=== TEMPORAL OVERLAP AUDIT ===")
print("Blocks crossing train boundary:", train_crossing)
print("Blocks crossing validation boundary:", val_crossing)

=== PROVISIONAL CHRONOLOGICAL SPLIT ===
Label       Anomaly  Normal
split                      
test           1116   85144
train         13115  389427
validation     2607   83652

=== EVENT COUNT STATISTICS ===
count    575061.000000
mean         19.433815
std           5.177735
min           2.000000
25%          19.000000
50%          19.000000
75%          20.000000
max         298.000000
Name: event_count, dtype: float64

=== SPLIT BOUNDARIES ===
train | blocks: 402542 | first: 20081109203518 | last: 20081111111628
validation | blocks: 86259 | first: 20081111060046 | last: 20081111111615
test | blocks: 86260 | first: 20081111083310 | last: 20081111111557

=== TEMPORAL OVERLAP AUDIT ===
Blocks crossing train boundary: 45396
Blocks crossing validation boundary: 37487


In [8]:
# Convert timestamp strings to datetime
metadata["start_dt"] = pd.to_datetime(
    metadata["first_timestamp"],
    format="%Y%m%d%H%M%S"
)

metadata["end_dt"] = pd.to_datetime(
    metadata["last_timestamp"],
    format="%Y%m%d%H%M%S"
)

# Calculate block duration
metadata["duration_seconds"] = (
    metadata["end_dt"] - metadata["start_dt"]
).dt.total_seconds()

print("=== BLOCK DURATION STATISTICS ===")
print(
    metadata["duration_seconds"].describe(
        percentiles=[0.25, 0.50, 0.75, 0.90, 0.95, 0.99]
    )
)

print("\n=== LONG-RUNNING BLOCKS ===")

for hours in [1, 3, 6, 12, 24]:
    count = (metadata["duration_seconds"] > hours * 3600).sum()
    print(f"Blocks lasting over {hours} hours: {count:,}")

print("\n=== DURATION BY SPLIT ===")
print(
    metadata.groupby("split")["duration_seconds"]
    .agg(["count", "median", "mean", "max"])
)

print("\n=== DURATION VALIDITY ===")
print("Negative durations:", (metadata["duration_seconds"] < 0).sum())
print("Zero-duration blocks:", (metadata["duration_seconds"] == 0).sum())

=== BLOCK DURATION STATISTICS ===
count    575061.000000
mean      16789.470527
std       17886.993688
min           0.000000
25%        1144.000000
50%        7229.000000
75%       33680.000000
90%       48159.000000
95%       50815.000000
99%       53186.000000
max       54025.000000
Name: duration_seconds, dtype: float64

=== LONG-RUNNING BLOCKS ===
Blocks lasting over 1 hours: 356,738
Blocks lasting over 3 hours: 248,517
Blocks lasting over 6 hours: 216,997
Blocks lasting over 12 hours: 68,708
Blocks lasting over 24 hours: 0

=== DURATION BY SPLIT ===
             count   median          mean      max
split                                             
test         86260     63.0    948.785126   9351.0
train       402542  22370.0  23127.381063  54025.0
validation   86259   2454.0   3053.423735  11724.0

=== DURATION VALIDITY ===
Negative durations: 0
Zero-duration blocks: 2683


In [9]:
# Existing provisional boundaries
t1 = metadata.loc[train_end, "first_timestamp"]
t2 = metadata.loc[val_end, "first_timestamp"]

# Strict temporal eligibility
train_mask = metadata["last_timestamp"] < t1

val_mask = (
    (metadata["first_timestamp"] >= t1) &
    (metadata["last_timestamp"] < t2)
)

test_mask = metadata["first_timestamp"] >= t2

# Construct candidate split
candidate = metadata.copy()
candidate["strict_split"] = "excluded"

candidate.loc[train_mask, "strict_split"] = "train"
candidate.loc[val_mask, "strict_split"] = "validation"
candidate.loc[test_mask, "strict_split"] = "test"

print("=== STRICT CHRONOLOGICAL SPLIT ===")
print(
    pd.crosstab(
        candidate["strict_split"],
        candidate["Label"]
    )
)

print("\n=== RETENTION ===")
print(candidate["strict_split"].value_counts())

retained = (candidate["strict_split"] != "excluded").sum()

print(
    f"\nRetained: {retained:,} / {len(candidate):,}"
    f" ({retained / len(candidate) * 100:.2f}%)"
)

print("\n=== ANOMALY RATES ===")
for split in ["train", "validation", "test", "excluded"]:
    subset = candidate[candidate["strict_split"] == split]

    anomaly_rate = (subset["Label"] == "Anomaly").mean()

    print(
        f"{split}: {len(subset):,} blocks | "
        f"{anomaly_rate * 100:.2f}% anomalies"
    )

print("\n=== TEMPORAL INTEGRITY ===")
print(
    "Train ends before validation:",
    candidate.loc[train_mask, "last_timestamp"].max() < t1
)

print(
    "Validation ends before test:",
    candidate.loc[val_mask, "last_timestamp"].max() < t2
)

=== STRICT CHRONOLOGICAL SPLIT ===
Label         Anomaly  Normal
strict_split                 
excluded         3077   78124
test             1116   85152
train           11674  345462
validation        971   49485

=== RETENTION ===
strict_split
train         357136
test           86268
excluded       81201
validation     50456
Name: count, dtype: int64

Retained: 493,860 / 575,061 (85.88%)

=== ANOMALY RATES ===
train: 357,136 blocks | 3.27% anomalies
validation: 50,456 blocks | 1.92% anomalies
test: 86,268 blocks | 1.29% anomalies
excluded: 81,201 blocks | 3.79% anomalies

=== TEMPORAL INTEGRITY ===
Train ends before validation: True
Validation ends before test: True


In [10]:
import json
import hashlib
from pathlib import Path
from google.colab import files

OUTPUT_DIR = Path("/content/logsentinel_artifacts")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Save only the columns required for reproducibility
manifest = candidate[
    [
        "BlockId",
        "Label",
        "first_timestamp",
        "last_timestamp",
        "first_line",
        "event_count",
        "duration_seconds",
        "strict_split"
    ]
].copy()

manifest_path = OUTPUT_DIR / "hdfs_split_manifest.csv"

manifest.to_csv(manifest_path, index=False)

# Verify split integrity
assert len(manifest) == 575061
assert manifest["BlockId"].is_unique
assert manifest["Label"].notna().all()

assert (
    manifest.loc[
        manifest["strict_split"] == "train",
        "last_timestamp"
    ].max() < t1
)

assert (
    manifest.loc[
        manifest["strict_split"] == "validation",
        "last_timestamp"
    ].max() < t2
)

# Save experiment configuration
split_config = {
    "dataset": "HDFS_v1",
    "strategy": "strict_chronological_time_purged",
    "train_boundary": t1,
    "validation_test_boundary": t2,
    "excluded_boundary_crossing_blocks": 81201,
    "random_seed": 42,
    "status": "frozen"
}

config_path = OUTPUT_DIR / "split_config.json"

with open(config_path, "w") as f:
    json.dump(split_config, f, indent=4)

# Generate file hash
sha256 = hashlib.sha256()

with open(manifest_path, "rb") as f:
    for chunk in iter(lambda: f.read(1024 * 1024), b""):
        sha256.update(chunk)

print("=== SPLIT FROZEN ===")
print("Manifest:", manifest_path)
print("Configuration:", config_path)
print("SHA256:", sha256.hexdigest())

print("\nFinal distribution:")
print(
    pd.crosstab(
        manifest["strict_split"],
        manifest["Label"]
    )
)

=== SPLIT FROZEN ===
Manifest: /content/logsentinel_artifacts/hdfs_split_manifest.csv
Configuration: /content/logsentinel_artifacts/split_config.json
SHA256: 5a7be4f2e0ce4464a511295a139e3a0bff3e22271eb982fd3a86c89bd8113a2d

Final distribution:
Label         Anomaly  Normal
strict_split                 
excluded         3077   78124
test             1116   85152
train           11674  345462
validation        971   49485


In [11]:
import re
from collections import Counter

# Precompile patterns for efficiency
PATTERNS = [
    # HDFS block identifiers
    (re.compile(r"blk_-?\d+"), "<BLOCK>"),

    # IP addresses with optional ports
    (re.compile(r"/?\b(?:\d{1,3}\.){3}\d{1,3}(?::\d+)?"), "<ADDRESS>"),

    # Hadoop job/task identifiers
    (re.compile(r"\b(?:job|task|attempt)_\d+(?:_\d+)*\b"), "<JOB>"),

    # Filesystem paths
    (re.compile(r"/[^\s,]+"), "<PATH>"),

    # Remaining numerical values
    (re.compile(r"(?<![\w])[-+]?\d+(?![\w])"), "<NUM>")
]

def extract_template(message):
    """Normalize variable fields while preserving event structure."""
    template = message

    for pattern, replacement in PATTERNS:
        template = pattern.sub(replacement, template)

    return " ".join(template.split())


def parse_log_line(line):
    """Parse one HDFS log line into structured fields."""
    parts = line.rstrip("\n").split(maxsplit=5)

    if len(parts) != 6:
        return None

    date, time, pid, level, component, message = parts

    block_match = re.search(r"blk_-?\d+", message)

    return {
        "timestamp": "20" + date + time,
        "pid": pid,
        "level": level,
        "component": component.rstrip(":"),
        "block_id": block_match.group() if block_match else None,
        "message": message,
        "template": extract_template(message)
    }


# Inspect a small sample before processing the full dataset
sample_records = []

with open(RAW_LOG, "r", encoding="utf-8", errors="replace") as f:
    for line in f:
        record = parse_log_line(line)

        if record is not None:
            sample_records.append(record)

        if len(sample_records) == 10000:
            break

sample_df = pd.DataFrame(sample_records)

print("=== PARSER SAMPLE AUDIT ===")
print("Parsed records:", len(sample_df))
print("Unique templates:", sample_df["template"].nunique())
print("Unique components:", sample_df["component"].nunique())
print("Missing block IDs:", sample_df["block_id"].isna().sum())

print("\n=== FIRST 10 EXTRACTED TEMPLATES ===")
for template in sample_df["template"].drop_duplicates().head(10):
    print("-", template)

print("\n=== TOP 15 MOST FREQUENT TEMPLATES ===")
print(sample_df["template"].value_counts().head(15))

=== PARSER SAMPLE AUDIT ===
Parsed records: 10000
Unique templates: 15
Unique components: 6
Missing block IDs: 0

=== FIRST 10 EXTRACTED TEMPLATES ===
- Receiving block <BLOCK> src: <ADDRESS> dest: <ADDRESS>
- BLOCK* NameSystem.allocateBlock: <PATH> <BLOCK>
- PacketResponder <NUM> for block <BLOCK> terminating
- Received block <BLOCK> of size <NUM> from <ADDRESS>
- BLOCK* NameSystem.addStoredBlock: blockMap updated: <ADDRESS> is added to <BLOCK> size <NUM>
- Received block <BLOCK> src: <ADDRESS> dest: <ADDRESS> of size <NUM>
- <ADDRESS>:Transmitted block <BLOCK> to <ADDRESS>
- <ADDRESS> Starting thread to transfer block <BLOCK> to <ADDRESS>, <ADDRESS>
- BLOCK* ask <ADDRESS> to replicate <BLOCK> to datanode(s) <ADDRESS> <ADDRESS>
- <ADDRESS> Served block <BLOCK> to <ADDRESS>

=== TOP 15 MOST FREQUENT TEMPLATES ===
template
Receiving block <BLOCK> src: <ADDRESS> dest: <ADDRESS>                                          3031
PacketResponder <NUM> for block <BLOCK> terminating              

In [12]:
from collections import Counter

# Use anomaly labels only to select diagnostic examples.
# These labels will NOT become HMM input features.
train_anomaly_blocks = set(
    manifest.loc[
        (manifest["strict_split"] == "train") &
        (manifest["Label"] == "Anomaly"),
        "BlockId"
    ]
)

sampled_templates = Counter()
anomaly_templates = Counter()
components = Counter()

sampled_count = 0
anomaly_sample_count = 0
parse_failures = 0

with open(RAW_LOG, "r", encoding="utf-8", errors="replace") as f:

    for line_number, line in enumerate(f):

        # Systematic sample spread throughout the full file
        take_regular_sample = line_number % 1000 == 0

        block_match = block_pattern.search(line)

        take_anomaly_sample = (
            block_match is not None
            and block_match.group() in train_anomaly_blocks
            and anomaly_sample_count < 10000
        )

        if not (take_regular_sample or take_anomaly_sample):
            continue

        record = parse_log_line(line)

        if record is None:
            parse_failures += 1
            continue

        if take_regular_sample:
            sampled_templates[record["template"]] += 1
            components[record["component"]] += 1
            sampled_count += 1

        if take_anomaly_sample:
            anomaly_templates[record["template"]] += 1
            anomaly_sample_count += 1


print("=== EXTENDED PARSER AUDIT ===")
print("Timeline sample size:", sampled_count)
print("Anomalous training-event sample:", anomaly_sample_count)
print("Parse failures:", parse_failures)

print("\nUnique templates (timeline sample):", len(sampled_templates))
print("Unique templates (anomalous training sample):", len(anomaly_templates))
print("Unique components:", len(components))

print("\n=== COMPONENT DISTRIBUTION ===")
for component, count in components.most_common():
    print(f"{component}: {count}")

print("\n=== ANOMALOUS TRAINING EVENT TEMPLATES ===")
for template, count in anomaly_templates.most_common(30):
    print(f"{count:5d} | {template}")

print("\n=== RARE TEMPLATES ===")
for template, count in sorted(
    sampled_templates.items(),
    key=lambda item: (item[1], item[0])
)[:20]:
    print(f"{count:5d} | {template}")

=== EXTENDED PARSER AUDIT ===
Timeline sample size: 11176
Anomalous training-event sample: 10000
Parse failures: 0

Unique templates (timeline sample): 20
Unique templates (anomalous training sample): 33
Unique components: 8

=== COMPONENT DISTRIBUTION ===
dfs.FSNamesystem: 3710
dfs.DataNode$PacketResponder: 3454
dfs.DataNode$DataXceiver: 2470
dfs.FSDataset: 1401
dfs.DataBlockScanner: 122
dfs.DataNode: 9
dfs.DataNode$DataTransfer: 8
dfs.DataNode$BlockReceiver: 2

=== ANOMALOUS TRAINING EVENT TEMPLATES ===
 2529 | Receiving block <BLOCK> src: <ADDRESS> dest: <ADDRESS>
 1877 | BLOCK* NameSystem.addStoredBlock: blockMap updated: <ADDRESS> is added to <BLOCK> size <NUM>
 1809 | PacketResponder <NUM> for block <BLOCK> terminating
 1801 | Received block <BLOCK> of size <NUM> from <ADDRESS>
 1022 | BLOCK* NameSystem.allocateBlock: <PATH> <BLOCK>
  203 | <ADDRESS> Served block <BLOCK> to <ADDRESS>
  201 | writeBlock <BLOCK> received exception java.io.IOException: Could not read from stream
   

In [13]:
from collections import Counter

# Only blocks in the strict training partition
train_blocks = set(
    manifest.loc[
        manifest["strict_split"] == "train",
        "BlockId"
    ]
)

template_counts = Counter()
component_counts = Counter()

training_log_count = 0
parse_failures = 0

with open(RAW_LOG, "r", encoding="utf-8", errors="replace") as f:

    for line in f:

        # Extract block ID first to avoid unnecessary parsing
        match = block_pattern.search(line)

        if not match:
            parse_failures += 1
            continue

        block_id = match.group()

        if block_id not in train_blocks:
            continue

        record = parse_log_line(line)

        if record is None:
            parse_failures += 1
            continue

        # Component-aware event template
        event_template = (
            record["component"]
            + " | "
            + record["template"]
        )

        template_counts[event_template] += 1
        component_counts[record["component"]] += 1

        training_log_count += 1


print("=== TRAINING EVENT INVENTORY ===")
print("Training blocks:", len(train_blocks))
print("Training log messages:", training_log_count)
print("Distinct event templates:", len(template_counts))
print("Parse failures:", parse_failures)

print("\n=== COMPONENT COUNTS ===")
for component, count in component_counts.most_common():
    print(component, count)

print("\n=== TOP 20 TRAINING TEMPLATES ===")
for template, count in template_counts.most_common(20):
    print(f"{count:>10,} | {template}")

print("\n=== RARE TRAINING TEMPLATES ===")
for template, count in sorted(
    template_counts.items(),
    key=lambda x: (x[1], x[0])
)[:20]:
    print(f"{count:>10,} | {template}")

=== TRAINING EVENT INVENTORY ===
Training blocks: 357136
Training log messages: 7447601
Distinct event templates: 52
Parse failures: 0

=== COMPONENT COUNTS ===
dfs.FSNamesystem 2388670
dfs.DataNode$PacketResponder 2114694
dfs.DataNode$DataXceiver 1856624
dfs.FSDataset 968716
dfs.DataBlockScanner 111493
dfs.DataNode 3277
dfs.DataNode$DataTransfer 3255
dfs.DataNode$BlockReceiver 854
dfs.PendingReplicationBlocks$PendingReplicationMonitor 18

=== TOP 20 TRAINING TEMPLATES ===
 1,067,847 | dfs.DataNode$DataXceiver | Receiving block <BLOCK> src: <ADDRESS> dest: <ADDRESS>
 1,063,882 | dfs.FSNamesystem | BLOCK* NameSystem.addStoredBlock: blockMap updated: <ADDRESS> is added to <BLOCK> size <NUM>
 1,057,349 | dfs.DataNode$PacketResponder | PacketResponder <NUM> for block <BLOCK> terminating
 1,057,271 | dfs.DataNode$PacketResponder | Received block <BLOCK> of size <NUM> from <ADDRESS>
   965,319 | dfs.FSDataset | Deleting block <BLOCK> file <PATH>
   962,466 | dfs.FSNamesystem | BLOCK* NameSys

In [14]:
import numpy as np
import json
from pathlib import Path
from collections import Counter

# --------------------------------------------------
# 1. Freeze training-only event vocabulary
# --------------------------------------------------

# ID 0 is reserved for unknown events
event_to_id = {"<UNKNOWN>": 0}

# Sort templates for deterministic IDs
for event_template in sorted(template_counts.keys()):
    event_to_id[event_template] = len(event_to_id)

id_to_event = {
    str(event_id): template
    for template, event_id in event_to_id.items()
}

print("=== EVENT VOCABULARY ===")
print("Known templates:", len(event_to_id) - 1)
print("Total observation symbols:", len(event_to_id))

assert len(event_to_id) == 53
assert event_to_id["<UNKNOWN>"] == 0


# --------------------------------------------------
# 2. Prepare retained block sequences
# --------------------------------------------------

retained_metadata = manifest[
    manifest["strict_split"].isin(
        ["train", "validation", "test"]
    )
].copy().reset_index(drop=True)

block_to_index = {
    block_id: i
    for i, block_id in enumerate(retained_metadata["BlockId"])
}

sequences = [[] for _ in range(len(retained_metadata))]

unknown_counts = Counter()
processed_counts = Counter()


# --------------------------------------------------
# 3. Construct sequences from raw logs
# --------------------------------------------------

with open(RAW_LOG, "r", encoding="utf-8", errors="replace") as f:

    for line in f:

        block_match = block_pattern.search(line)

        if block_match is None:
            continue

        block_id = block_match.group()

        # Exclude boundary-crossing blocks
        index = block_to_index.get(block_id)

        if index is None:
            continue

        record = parse_log_line(line)

        event_template = (
            record["component"]
            + " | "
            + record["template"]
        )

        event_id = event_to_id.get(event_template, 0)

        sequences[index].append(event_id)

        split = retained_metadata.iloc[index]["strict_split"]
        processed_counts[split] += 1

        if event_id == 0:
            unknown_counts[split] += 1


# --------------------------------------------------
# 4. Verify sequence construction
# --------------------------------------------------

sequence_lengths = np.array(
    [len(seq) for seq in sequences],
    dtype=np.int32
)

expected_lengths = retained_metadata[
    "event_count"
].to_numpy()

assert np.array_equal(
    sequence_lengths,
    expected_lengths
), "Sequence lengths do not match raw-log audit!"

assert (sequence_lengths > 0).all()

print("\n=== SEQUENCE VERIFICATION ===")
print("Total retained sequences:", len(sequences))
print("Total retained events:", int(sequence_lengths.sum()))
print("Minimum length:", sequence_lengths.min())
print("Median length:", np.median(sequence_lengths))
print("Maximum length:", sequence_lengths.max())

print("\n=== UNKNOWN EVENT STATISTICS ===")

for split in ["train", "validation", "test"]:

    total = processed_counts[split]
    unknown = unknown_counts[split]

    rate = 100 * unknown / total if total else 0

    print(
        f"{split}: {unknown:,} / {total:,} "
        f"({rate:.4f}%)"
    )


# --------------------------------------------------
# 5. Save compact sequence representation
# --------------------------------------------------

# Flatten sequences into a compact array
flat_events = np.fromiter(
    (event for seq in sequences for event in seq),
    dtype=np.uint16,
    count=int(sequence_lengths.sum())
)

offsets = np.zeros(
    len(sequences) + 1,
    dtype=np.int64
)

offsets[1:] = np.cumsum(sequence_lengths)

assert offsets[-1] == len(flat_events)

output_dir = Path("/content/logsentinel_artifacts")
output_dir.mkdir(exist_ok=True)

np.savez_compressed(
    output_dir / "hdfs_event_sequences.npz",
    events=flat_events,
    offsets=offsets
)

retained_metadata.to_csv(
    output_dir / "retained_block_metadata.csv",
    index=False
)

with open(output_dir / "event_vocabulary.json", "w") as f:
    json.dump(event_to_id, f, indent=2)

print("\n=== ARTIFACTS SAVED ===")
print("hdfs_event_sequences.npz")
print("retained_block_metadata.csv")
print("event_vocabulary.json")

print("\n=== SAMPLE SEQUENCE ===")

sample_index = 0

print("Block ID:", retained_metadata.iloc[sample_index]["BlockId"])
print("Split:", retained_metadata.iloc[sample_index]["strict_split"])
print("Sequence length:", len(sequences[sample_index]))
print("Event IDs:", sequences[sample_index][:30])

=== EVENT VOCABULARY ===
Known templates: 52
Total observation symbols: 53

=== SEQUENCE VERIFICATION ===
Total retained sequences: 493860
Total retained events: 9625001
Minimum length: 2
Median length: 19.0
Maximum length: 298

=== UNKNOWN EVENT STATISTICS ===
train: 0 / 7,447,601 (0.0000%)
validation: 0 / 828,907 (0.0000%)
test: 0 / 1,348,493 (0.0000%)

=== ARTIFACTS SAVED ===
hdfs_event_sequences.npz
retained_block_metadata.csv
event_vocabulary.json

=== SAMPLE SEQUENCE ===
Block ID: blk_-1608999687919862906
Split: train
Sequence length: 269
Event IDs: [18, 47, 18, 18, 39, 39, 40, 40, 39, 40, 46, 46, 46, 17, 18, 14, 17, 18, 4, 51, 46, 46, 15, 51, 17, 17, 18, 18, 14, 4]


In [15]:
!pip -q install hmmlearn==0.3.3

In [16]:
import numpy as np
import time

from hmmlearn.hmm import CategoricalHMM

# --------------------------------------------------
# 1. Recover our processed sequences
# --------------------------------------------------

data = np.load(
    "/content/logsentinel_artifacts/hdfs_event_sequences.npz"
)

events = data["events"]
offsets = data["offsets"]

# Verify loaded artifacts
assert len(offsets) == len(retained_metadata) + 1
assert offsets[-1] == len(events)

# --------------------------------------------------
# 2. Select normal training blocks only
# --------------------------------------------------

normal_train_indices = np.flatnonzero(
    (
        retained_metadata["strict_split"].to_numpy() == "train"
    ) &
    (
        retained_metadata["Label"].to_numpy() == "Normal"
    )
)

print("Available normal training blocks:", len(normal_train_indices))

# Pilot: 20,000 sequences sampled across the training period
pilot_size = 20000

positions = np.linspace(
    0,
    len(normal_train_indices) - 1,
    pilot_size,
    dtype=int
)

pilot_indices = normal_train_indices[positions]

# --------------------------------------------------
# 3. Prepare HMM input
# --------------------------------------------------

pilot_sequences = [
    events[offsets[i]:offsets[i + 1]]
    for i in pilot_indices
]

X_pilot = np.concatenate(pilot_sequences).reshape(-1, 1)

lengths_pilot = [
    len(seq) for seq in pilot_sequences
]

print("\n=== PILOT DATA ===")
print("Sequences:", len(lengths_pilot))
print("Observations:", len(X_pilot))
print("Input shape:", X_pilot.shape)

# --------------------------------------------------
# 4. Train categorical HMM
# --------------------------------------------------

hmm_pilot = CategoricalHMM(
    n_components=3,
    n_features=53,
    n_iter=15,
    tol=1e-3,
    random_state=42,
    implementation="scaling",
    startprob_prior=1.1,
    transmat_prior=1.1,
    emissionprob_prior=1.1,
    verbose=False
)

start_time = time.time()

hmm_pilot.fit(
    X_pilot,
    lengths=lengths_pilot
)

training_time = time.time() - start_time

# --------------------------------------------------
# 5. Inspect training results
# --------------------------------------------------

print("\n=== HMM PILOT RESULTS ===")
print("Training time:", round(training_time, 2), "seconds")
print("Converged:", hmm_pilot.monitor_.converged)
print("Iterations:", hmm_pilot.monitor_.iter)

print("\nLog-likelihood history:")
print(list(hmm_pilot.monitor_.history))

print("\nInitial probabilities:")
print(hmm_pilot.startprob_)

print("\nTransition matrix:")
print(hmm_pilot.transmat_)

print("\n=== PROBABILITY CHECKS ===")

print(
    "Initial probabilities sum:",
    hmm_pilot.startprob_.sum()
)

print(
    "Transition row sums:",
    hmm_pilot.transmat_.sum(axis=1)
)

print(
    "Emission row sums:",
    hmm_pilot.emissionprob_.sum(axis=1)
)

assert np.allclose(hmm_pilot.startprob_.sum(), 1.0)
assert np.allclose(hmm_pilot.transmat_.sum(axis=1), 1.0)
assert np.allclose(hmm_pilot.emissionprob_.sum(axis=1), 1.0)

assert np.isfinite(hmm_pilot.transmat_).all()
assert np.isfinite(hmm_pilot.emissionprob_).all()

print("\nProbability validation: PASSED")

Available normal training blocks: 345462

=== PILOT DATA ===
Sequences: 20000
Observations: 419676
Input shape: (419676, 1)

=== HMM PILOT RESULTS ===
Training time: 11.63 seconds
Converged: True
Iterations: 15

Log-likelihood history:
[-1774068.5771108617, -861703.6392598293, -783298.606874396, -758801.082599649, -753798.4106273854, -751612.0262218023, -750055.4528689185, -748549.2587303101, -746268.3009349399, -741581.0882579782, -734079.0644555165, -728845.9381455149, -726587.8999412743, -724925.5919739556, -723014.2839846513]

Initial probabilities:
[7.99883293e-01 2.00111705e-01 5.00182252e-06]

Transition matrix:
[[0.01112146 0.98453187 0.00434666]
 [0.81940355 0.02513758 0.15545887]
 [0.15415385 0.00162392 0.84422223]]

=== PROBABILITY CHECKS ===
Initial probabilities sum: 1.0
Transition row sums: [1. 1. 1.]
Emission row sums: [1. 1. 1.]

Probability validation: PASSED


In [17]:
import numpy as np
import time

# Continue from the current trained parameters
hmm_pilot.init_params = ""

# Allow additional EM iterations
hmm_pilot.n_iter = 60
hmm_pilot.tol = 1e-3

start_time = time.time()

hmm_pilot.fit(
    X_pilot,
    lengths=lengths_pilot
)

elapsed = time.time() - start_time

history = np.array(
    list(hmm_pilot.monitor_.history)
)

improvements = np.diff(history)

print("=== EXTENDED HMM TRAINING ===")
print("Additional training time:", round(elapsed, 2), "seconds")
print("Additional iterations:", len(history))
print("Final log-likelihood:", history[-1])

print("\n=== LAST 10 LOG-LIKELIHOODS ===")
for value in history[-10:]:
    print(round(value, 4))

print("\n=== CONVERGENCE ANALYSIS ===")

if len(improvements) > 0:
    final_improvement = improvements[-1]

    relative_improvement = (
        abs(final_improvement) /
        max(abs(history[-2]), 1)
    )

    print("Final absolute improvement:", final_improvement)
    print("Final relative improvement:", relative_improvement)

    print(
        "Absolute convergence criterion met:",
        0 <= final_improvement < hmm_pilot.tol
    )

print("\n=== PARAMETER VALIDATION ===")

assert np.allclose(hmm_pilot.startprob_.sum(), 1.0)
assert np.allclose(hmm_pilot.transmat_.sum(axis=1), 1.0)
assert np.allclose(hmm_pilot.emissionprob_.sum(axis=1), 1.0)

assert np.isfinite(history).all()
assert np.isfinite(hmm_pilot.transmat_).all()
assert np.isfinite(hmm_pilot.emissionprob_).all()

print("Parameter validation: PASSED")

=== EXTENDED HMM TRAINING ===
Additional training time: 11.04 seconds
Additional iterations: 15
Final log-likelihood: -694017.0763638215

=== LAST 10 LOG-LIKELIHOODS ===
-694376.9031
-694199.0814
-694108.6007
-694066.6547
-694046.4085
-694035.6318
-694029.0686
-694024.4301
-694020.6679
-694017.0764

=== CONVERGENCE ANALYSIS ===
Final absolute improvement: 3.591500152135268
Final relative improvement: 5.174918152205798e-06
Absolute convergence criterion met: False

=== PARAMETER VALIDATION ===
Parameter validation: PASSED


In [18]:
# Update both the model and its convergence monitor
hmm_pilot.n_iter = 60
hmm_pilot.monitor_.n_iter = 60

# Keep learned parameters; do not reinitialize
hmm_pilot.init_params = ""

# Absolute log-likelihood tolerance
hmm_pilot.tol = 1e-3
hmm_pilot.monitor_.tol = 1e-3

start_time = time.time()

hmm_pilot.fit(
    X_pilot,
    lengths=lengths_pilot
)

elapsed = time.time() - start_time

history = np.array(list(hmm_pilot.monitor_.history))

print("=== FINAL PILOT CONVERGENCE CHECK ===")
print("Training time:", round(elapsed, 2), "seconds")
print("Iterations executed:", len(history))
print("Final log-likelihood:", history[-1])

print("\nLast 5 log-likelihoods:")
print(history[-5:])

if len(history) > 1:
    improvement = history[-1] - history[-2]

    print("\nAbsolute improvement:", improvement)
    print(
        "Relative improvement:",
        improvement / max(abs(history[-2]), 1)
    )

    print(
        "Strict tolerance satisfied:",
        0 <= improvement < 1e-3
    )

print("\nMonitor iteration limit:", hmm_pilot.monitor_.n_iter)

=== FINAL PILOT CONVERGENCE CHECK ===
Training time: 43.9 seconds
Iterations executed: 55
Final log-likelihood: -693996.01485557

Last 5 log-likelihoods:
[-693996.01919023 -693996.01801724 -693996.01690754 -693996.01585524
 -693996.01485557]

Absolute improvement: 0.0009996745502576232
Relative improvement: 1.4404615119089355e-09
Strict tolerance satisfied: True

Monitor iteration limit: 60


In [19]:
import numpy as np

def scaled_forward(observations, startprob, transmat, emissionprob):
    """
    Compute HMM sequence log-likelihood using the
    numerically stable scaled Forward algorithm.

    Parameters
    ----------
    observations : 1D array of integer event IDs
    startprob : shape (N,)
    transmat : shape (N, N)
    emissionprob : shape (N, M)

    Returns
    -------
    log_likelihood : float
    """

    obs = np.asarray(observations, dtype=np.int64)

    if obs.ndim != 1 or len(obs) == 0:
        raise ValueError("Observations must be a non-empty 1D array.")

    N = len(startprob)

    if np.any(obs < 0) or np.any(obs >= emissionprob.shape[1]):
        raise ValueError("Observation ID is outside the vocabulary.")

    # Initialization: alpha_1(j) = pi_j * B_j(o_1)
    alpha = startprob * emissionprob[:, obs[0]]

    scale = alpha.sum()

    if scale <= 0:
        return float("-inf")

    alpha = alpha / scale
    log_likelihood = np.log(scale)

    # Recursion
    for t in range(1, len(obs)):

        # alpha_t(j) = B_j(o_t) * sum_i(alpha_(t-1)(i) * A_ij)
        alpha = (alpha @ transmat) * emissionprob[:, obs[t]]

        scale = alpha.sum()

        if scale <= 0:
            return float("-inf")

        alpha = alpha / scale
        log_likelihood += np.log(scale)

    return float(log_likelihood)


print("Custom scaled Forward algorithm defined successfully.")

Custom scaled Forward algorithm defined successfully.


In [20]:
# Test on 50 pilot sequences
test_sequences = pilot_sequences[:50]

differences = []

for i, seq in enumerate(test_sequences):

    # Our implementation
    custom_ll = scaled_forward(
        seq,
        hmm_pilot.startprob_,
        hmm_pilot.transmat_,
        hmm_pilot.emissionprob_
    )

    # Reference implementation
    reference_ll = hmm_pilot.score(
        np.asarray(seq).reshape(-1, 1)
    )

    difference = abs(custom_ll - reference_ll)

    differences.append(difference)

print("=== SCALED FORWARD VALIDATION ===")

print("Sequences tested:", len(differences))
print("Maximum absolute difference:", max(differences))
print("Mean absolute difference:", np.mean(differences))

print("\nFirst 5 comparisons:")

for i, seq in enumerate(test_sequences[:5]):

    custom = scaled_forward(
        seq,
        hmm_pilot.startprob_,
        hmm_pilot.transmat_,
        hmm_pilot.emissionprob_
    )

    reference = hmm_pilot.score(
        np.asarray(seq).reshape(-1, 1)
    )

    print(
        f"Sequence {i + 1}: "
        f"Custom={custom:.8f}, "
        f"Reference={reference:.8f}"
    )

# Numerical agreement requirement
assert np.all(
    np.array(differences) < 1e-6
), "Custom Forward algorithm differs from reference!"

print("\nCUSTOM SCALED FORWARD VALIDATION: PASSED")

=== SCALED FORWARD VALIDATION ===
Sequences tested: 50
Maximum absolute difference: 1.4210854715202004e-14
Mean absolute difference: 9.947598300641404e-16

First 5 comparisons:
Sequence 1: Custom=-707.07259969, Reference=-707.07259969
Sequence 2: Custom=-75.00725533, Reference=-75.00725533
Sequence 3: Custom=-77.53827203, Reference=-77.53827203
Sequence 4: Custom=-75.00725533, Reference=-75.00725533
Sequence 5: Custom=-75.00725533, Reference=-75.00725533

CUSTOM SCALED FORWARD VALIDATION: PASSED


In [21]:
import time
import numpy as np
from hmmlearn.hmm import CategoricalHMM

# Preserve our existing converged 3-state model
candidate_models = {
    3: hmm_pilot
}

training_summary = []

# Train additional candidates
for n_states in [5, 8]:

    print(f"\n{'=' * 50}")
    print(f"TRAINING {n_states}-STATE HMM")
    print(f"{'=' * 50}")

    model = CategoricalHMM(
        n_components=n_states,
        n_features=len(event_to_id),
        n_iter=100,
        tol=1e-3,
        random_state=42,
        implementation="scaling",
        startprob_prior=1.1,
        transmat_prior=1.1,
        emissionprob_prior=1.1,
        verbose=False
    )

    start = time.time()

    model.fit(
        X_pilot,
        lengths=lengths_pilot
    )

    elapsed = time.time() - start

    history = np.array(
        list(model.monitor_.history)
    )

    final_improvement = (
        history[-1] - history[-2]
        if len(history) > 1
        else float("nan")
    )

    relative_improvement = (
        abs(final_improvement) / max(abs(history[-2]), 1)
        if len(history) > 1
        else float("nan")
    )

    # Validate model parameters
    assert np.allclose(model.startprob_.sum(), 1)
    assert np.allclose(model.transmat_.sum(axis=1), 1)
    assert np.allclose(model.emissionprob_.sum(axis=1), 1)

    assert np.isfinite(history).all()
    assert np.isfinite(model.transmat_).all()
    assert np.isfinite(model.emissionprob_).all()

    candidate_models[n_states] = model

    training_summary.append({
        "states": n_states,
        "iterations": len(history),
        "training_seconds": round(elapsed, 2),
        "final_log_likelihood": float(history[-1]),
        "absolute_improvement": float(final_improvement),
        "relative_improvement": float(relative_improvement),
        "strict_convergence": bool(
            0 <= final_improvement < model.tol
        )
    })

    print("Training time:", round(elapsed, 2), "seconds")
    print("Iterations:", len(history))
    print("Final log-likelihood:", round(history[-1], 4))
    print("Final absolute improvement:", final_improvement)
    print("Final relative improvement:", relative_improvement)
    print(
        "Strict convergence:",
        0 <= final_improvement < model.tol
    )

print("\n=== MODEL TRAINING SUMMARY ===")

import pandas as pd

summary_df = pd.DataFrame(training_summary)

print(summary_df.to_string(index=False))


TRAINING 5-STATE HMM
Training time: 86.49 seconds
Iterations: 100
Final log-likelihood: -452311.032
Final absolute improvement: 0.0012293667532503605
Final relative improvement: 2.717967643780136e-09
Strict convergence: False

TRAINING 8-STATE HMM
Training time: 102.42 seconds
Iterations: 100
Final log-likelihood: -371107.9699
Final absolute improvement: 0.11451538401888683
Final relative improvement: 3.085769047234631e-07
Strict convergence: False

=== MODEL TRAINING SUMMARY ===
 states  iterations  training_seconds  final_log_likelihood  absolute_improvement  relative_improvement  strict_convergence
      5         100             86.49        -452311.032003              0.001229          2.717968e-09               False
      8         100            102.42        -371107.969939              0.114515          3.085769e-07               False


In [22]:
import numpy as np
import pandas as pd
import time

from sklearn.metrics import (
    precision_recall_curve,
    average_precision_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix
)

# --------------------------------------------------
# 1. Identify validation sequences
# --------------------------------------------------

validation_indices = np.flatnonzero(
    retained_metadata["strict_split"].to_numpy() == "validation"
)

y_val = (
    retained_metadata.iloc[validation_indices]["Label"]
    .eq("Anomaly")
    .astype(int)
    .to_numpy()
)

print("Validation sequences:", len(validation_indices))
print("Normal:", (y_val == 0).sum())
print("Anomaly:", (y_val == 1).sum())


# --------------------------------------------------
# 2. Score sequences with our custom Forward algorithm
# --------------------------------------------------

def score_sequences(model, indices):
    scores = np.empty(len(indices), dtype=np.float64)

    for j, i in enumerate(indices):
        seq = events[offsets[i]:offsets[i + 1]]

        ll = scaled_forward(
            seq,
            model.startprob_,
            model.transmat_,
            model.emissionprob_
        )

        # Length-normalized negative log-likelihood
        scores[j] = -ll / len(seq)

    return scores


# --------------------------------------------------
# 3. Evaluate candidate models
# --------------------------------------------------

validation_results = []
all_validation_scores = {}

for n_states in [3, 5, 8]:

    print(f"\nEvaluating {n_states}-state HMM...")

    model = candidate_models[n_states]

    start_time = time.time()

    scores = score_sequences(
        model,
        validation_indices
    )

    elapsed = time.time() - start_time

    assert np.isfinite(scores).all()

    all_validation_scores[n_states] = scores

    # Precision-recall curve
    precision, recall, thresholds = precision_recall_curve(
        y_val,
        scores
    )

    # Last PR point has no corresponding threshold
    f1_values = (
        2 * precision[:-1] * recall[:-1] /
        (precision[:-1] + recall[:-1] + 1e-12)
    )

    best_idx = int(np.argmax(f1_values))
    threshold = float(thresholds[best_idx])

    predictions = (scores >= threshold).astype(int)

    tn, fp, fn, tp = confusion_matrix(
        y_val,
        predictions,
        labels=[0, 1]
    ).ravel()

    result = {
        "states": n_states,
        "average_precision": average_precision_score(
            y_val, scores
        ),
        "precision": precision_score(
            y_val, predictions,
            zero_division=0
        ),
        "recall": recall_score(
            y_val, predictions,
            zero_division=0
        ),
        "f1": f1_score(
            y_val, predictions,
            zero_division=0
        ),
        "false_alarm_rate": fp / (fp + tn),
        "threshold": threshold,
        "TP": int(tp),
        "FP": int(fp),
        "TN": int(tn),
        "FN": int(fn),
        "scoring_seconds": round(elapsed, 2)
    }

    validation_results.append(result)

    print(
        f"AP={result['average_precision']:.4f} | "
        f"F1={result['f1']:.4f} | "
        f"FAR={result['false_alarm_rate']:.4f} | "
        f"Time={elapsed:.1f}s"
    )


# --------------------------------------------------
# 4. Compare and select
# --------------------------------------------------

validation_df = pd.DataFrame(validation_results)

validation_df = validation_df.sort_values(
    ["average_precision", "f1"],
    ascending=False
).reset_index(drop=True)

best_states = int(validation_df.iloc[0]["states"])

print("\n=== FINAL VALIDATION COMPARISON ===")

print(
    validation_df[
        [
            "states",
            "average_precision",
            "precision",
            "recall",
            "f1",
            "false_alarm_rate",
            "threshold"
        ]
    ].to_string(index=False)
)

print("\nSELECTED HMM:", best_states, "hidden states")


# --------------------------------------------------
# 5. Save validation results
# --------------------------------------------------

output_dir = Path("/content/logsentinel_artifacts")

validation_df.to_csv(
    output_dir / "hmm_validation_comparison.csv",
    index=False
)

np.savez_compressed(
    output_dir / "hmm_validation_scores.npz",
    labels=y_val,
    scores_3=all_validation_scores[3],
    scores_5=all_validation_scores[5],
    scores_8=all_validation_scores[8]
)

print("\nValidation artifacts saved.")

Validation sequences: 50456
Normal: 49485
Anomaly: 971

Evaluating 3-state HMM...
AP=0.7535 | F1=0.6603 | FAR=0.0003 | Time=8.3s

Evaluating 5-state HMM...
AP=0.6432 | F1=0.6769 | FAR=0.0021 | Time=6.3s

Evaluating 8-state HMM...
AP=0.6624 | F1=0.6510 | FAR=0.0024 | Time=8.1s

=== FINAL VALIDATION COMPARISON ===
 states  average_precision  precision   recall       f1  false_alarm_rate  threshold
      3           0.753532   0.973896 0.499485 0.660313          0.000263   3.024498
      8           0.662444   0.817757 0.540680 0.650961          0.002364   1.471441
      5           0.643167   0.843318 0.565396 0.676942          0.002061   1.542821

SELECTED HMM: 3 hidden states

Validation artifacts saved.


In [23]:
import numpy as np
import pandas as pd
import time
import joblib
import json

from pathlib import Path
from hmmlearn.hmm import CategoricalHMM

output_dir = Path("/content/logsentinel_artifacts")
output_dir.mkdir(exist_ok=True)

# --------------------------------------------------
# 1. Prepare complete normal training dataset
# --------------------------------------------------

train_indices = np.flatnonzero(
    (retained_metadata["strict_split"].to_numpy() == "train") &
    (retained_metadata["Label"].to_numpy() == "Normal")
)

train_lengths = (
    offsets[train_indices + 1] - offsets[train_indices]
).astype(int)

X_train = np.concatenate([
    events[offsets[i]:offsets[i + 1]]
    for i in train_indices
]).reshape(-1, 1)

print("=== FINAL TRAINING DATA ===")
print("Normal training blocks:", len(train_indices))
print("Training observations:", len(X_train))
print("Hidden states:", best_states)

assert len(train_indices) == 345462
assert train_lengths.sum() == len(X_train)

# --------------------------------------------------
# 2. Initialize from selected 3-state pilot
# --------------------------------------------------

final_hmm = CategoricalHMM(
    n_components=best_states,
    n_features=len(event_to_id),
    n_iter=35,
    tol=1e-3,
    random_state=42,
    implementation="scaling",
    startprob_prior=1.1,
    transmat_prior=1.1,
    emissionprob_prior=1.1,
    init_params="",
    params="ste",
    verbose=False
)

# Transfer pilot parameters as initialization
pilot_model = candidate_models[best_states]

final_hmm.startprob_ = pilot_model.startprob_.copy()
final_hmm.transmat_ = pilot_model.transmat_.copy()
final_hmm.emissionprob_ = pilot_model.emissionprob_.copy()

# --------------------------------------------------
# 3. Train on complete normal training partition
# --------------------------------------------------

print("\nStarting full training...")

start_time = time.time()

final_hmm.fit(
    X_train,
    lengths=train_lengths.tolist()
)

training_seconds = time.time() - start_time

history = np.array(
    list(final_hmm.monitor_.history)
)

# --------------------------------------------------
# 4. Validate trained model
# --------------------------------------------------

assert np.isfinite(history).all()

assert np.allclose(final_hmm.startprob_.sum(), 1)
assert np.allclose(final_hmm.transmat_.sum(axis=1), 1)
assert np.allclose(final_hmm.emissionprob_.sum(axis=1), 1)

assert np.isfinite(final_hmm.startprob_).all()
assert np.isfinite(final_hmm.transmat_).all()
assert np.isfinite(final_hmm.emissionprob_).all()

last_delta = (
    float(history[-1] - history[-2])
    if len(history) > 1 else None
)

relative_delta = (
    abs(last_delta) / max(abs(history[-2]), 1)
    if last_delta is not None else None
)

strict_convergence = (
    last_delta is not None
    and 0 <= last_delta < final_hmm.tol
)

print("\n=== FULL TRAINING RESULTS ===")
print("Training time:", round(training_seconds, 2), "seconds")
print("Iterations:", len(history))
print("Final log-likelihood:", history[-1])
print("Final absolute improvement:", last_delta)
print("Final relative improvement:", relative_delta)
print("Strict convergence:", strict_convergence)

# --------------------------------------------------
# 5. Save model and training metadata
# --------------------------------------------------

model_path = output_dir / "hmm_final_3state.joblib"

joblib.dump(final_hmm, model_path)

training_report = {
    "model": "CategoricalHMM",
    "hidden_states": int(best_states),
    "training_blocks": int(len(train_indices)),
    "training_observations": int(len(X_train)),
    "training_seconds": float(training_seconds),
    "iterations": int(len(history)),
    "final_log_likelihood": float(history[-1]),
    "last_absolute_improvement": last_delta,
    "last_relative_improvement": relative_delta,
    "strict_convergence": bool(strict_convergence),
    "initialization": "validated_3state_pilot",
    "random_seed": 42,
    "training_subset": "all_normal_training_blocks"
}

with open(output_dir / "hmm_training_report.json", "w") as f:
    json.dump(training_report, f, indent=2)

np.save(
    output_dir / "hmm_training_history.npy",
    history
)

print("\n=== SAVED ===")
print(model_path)
print(output_dir / "hmm_training_report.json")
print(output_dir / "hmm_training_history.npy")

=== FINAL TRAINING DATA ===
Normal training blocks: 345462
Training observations: 7249666
Hidden states: 3

Starting full training...

=== FULL TRAINING RESULTS ===
Training time: 480.75 seconds
Iterations: 35
Final log-likelihood: -11984189.624379812
Final absolute improvement: 0.058717500418424606
Final relative improvement: 4.899580361385691e-09
Strict convergence: False

=== SAVED ===
/content/logsentinel_artifacts/hmm_final_3state.joblib
/content/logsentinel_artifacts/hmm_training_report.json
/content/logsentinel_artifacts/hmm_training_history.npy


In [24]:
import numpy as np
import pandas as pd
import json
import time

from sklearn.metrics import (
    precision_recall_curve,
    average_precision_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix
)

# --------------------------------------------------
# 1. Prepare validation and test partitions
# --------------------------------------------------

split_array = retained_metadata["strict_split"].to_numpy()
label_array = retained_metadata["Label"].to_numpy()

val_indices = np.flatnonzero(split_array == "validation")
test_indices = np.flatnonzero(split_array == "test")

y_val = (label_array[val_indices] == "Anomaly").astype(int)
y_test = (label_array[test_indices] == "Anomaly").astype(int)

print("=== EVALUATION DATA ===")
print("Validation:", len(val_indices), "blocks")
print("Test:", len(test_indices), "blocks")

# --------------------------------------------------
# 2. Score validation sequences
# --------------------------------------------------

print("\nScoring validation partition...")
start = time.time()

val_scores = score_sequences(
    final_hmm,
    val_indices
)

print("Validation scoring time:", round(time.time() - start, 2))

assert np.isfinite(val_scores).all()

# --------------------------------------------------
# 3. Select threshold using validation data
# --------------------------------------------------

precision, recall, thresholds = precision_recall_curve(
    y_val,
    val_scores
)

f1_values = (
    2 * precision[:-1] * recall[:-1] /
    (precision[:-1] + recall[:-1] + 1e-12)
)

best_threshold_idx = int(np.argmax(f1_values))
final_threshold = float(thresholds[best_threshold_idx])

print("\n=== FINAL THRESHOLD ===")
print("Selected threshold:", final_threshold)
print("Validation F1:", f1_values[best_threshold_idx])

# --------------------------------------------------
# 4. Score the untouched test partition
# --------------------------------------------------

print("\nScoring test partition...")
start = time.time()

test_scores = score_sequences(
    final_hmm,
    test_indices
)

print("Test scoring time:", round(time.time() - start, 2))

assert np.isfinite(test_scores).all()

# Apply the validation-selected threshold without modification
test_predictions = (test_scores >= final_threshold).astype(int)
val_predictions = (val_scores >= final_threshold).astype(int)

# --------------------------------------------------
# 5. Calculate evaluation metrics
# --------------------------------------------------

def calculate_metrics(y_true, y_pred, scores):

    tn, fp, fn, tp = confusion_matrix(
        y_true, y_pred, labels=[0, 1]
    ).ravel()

    return {
        "precision": float(precision_score(
            y_true, y_pred, zero_division=0
        )),
        "recall": float(recall_score(
            y_true, y_pred, zero_division=0
        )),
        "f1": float(f1_score(
            y_true, y_pred, zero_division=0
        )),
        "false_alarm_rate": float(fp / (fp + tn)),
        "average_precision": float(
            average_precision_score(y_true, scores)
        ),
        "TP": int(tp),
        "FP": int(fp),
        "TN": int(tn),
        "FN": int(fn)
    }


val_metrics = calculate_metrics(
    y_val, val_predictions, val_scores
)

test_metrics = calculate_metrics(
    y_test, test_predictions, test_scores
)

print("\n=== FINAL VALIDATION METRICS ===")
for key, value in val_metrics.items():
    print(f"{key}: {value}")

print("\n=== FINAL TEST METRICS ===")
for key, value in test_metrics.items():
    print(f"{key}: {value}")

# --------------------------------------------------
# 6. Save predictions for the dashboard
# --------------------------------------------------

prediction_df = retained_metadata.iloc[test_indices][
    [
        "BlockId",
        "Label",
        "first_timestamp",
        "last_timestamp",
        "event_count"
    ]
].copy()

prediction_df["anomaly_score"] = test_scores
prediction_df["predicted_anomaly"] = test_predictions

prediction_df.to_csv(
    output_dir / "hmm_test_predictions.csv",
    index=False
)

np.savez_compressed(
    output_dir / "hmm_final_evaluation_scores.npz",
    validation_labels=y_val,
    validation_scores=val_scores,
    test_labels=y_test,
    test_scores=test_scores,
    test_predictions=test_predictions
)

evaluation_report = {
    "model": "CategoricalHMM",
    "hidden_states": 3,
    "threshold": final_threshold,
    "threshold_selection": "validation_f1_maximization",
    "anomaly_score": "negative_log_likelihood_per_event",
    "validation": val_metrics,
    "test": test_metrics
}

with open(output_dir / "hmm_final_evaluation.json", "w") as f:
    json.dump(evaluation_report, f, indent=2)

print("\n=== ARTIFACTS SAVED ===")
print("hmm_test_predictions.csv")
print("hmm_final_evaluation_scores.npz")
print("hmm_final_evaluation.json")

=== EVALUATION DATA ===
Validation: 50456 blocks
Test: 86268 blocks

Scoring validation partition...
Validation scoring time: 6.45

=== FINAL THRESHOLD ===
Selected threshold: 2.991257009736873
Validation F1: 0.6603131381887961

Scoring test partition...
Test scoring time: 12.26

=== FINAL VALIDATION METRICS ===
precision: 0.9738955823293173
recall: 0.49948506694129763
f1: 0.6603131381892444
false_alarm_rate: 0.0002627058704657977
average_precision: 0.7469111876097106
TP: 485
FP: 13
TN: 49472
FN: 486

=== FINAL TEST METRICS ===
precision: 0.6954492415402567
recall: 0.5340501792114696
f1: 0.6041561074505829
false_alarm_rate: 0.00306510710259301
average_precision: 0.6926897700385918
TP: 596
FP: 261
TN: 84891
FN: 520

=== ARTIFACTS SAVED ===
hmm_test_predictions.csv
hmm_final_evaluation_scores.npz
hmm_final_evaluation.json


In [25]:
import numpy as np
import pandas as pd
import json
import time

from sklearn.metrics import (
    precision_recall_curve,
    average_precision_score
)

M = len(event_to_id)
SMOOTHING = 1.0

# --------------------------------------------------
# 1. Fit baselines using normal training sequences
# --------------------------------------------------

event_counts = np.zeros(M, dtype=np.int64)
start_counts = np.zeros(M, dtype=np.int64)
transition_counts = np.zeros((M, M), dtype=np.int64)

start_time = time.time()

for i in train_indices:
    seq = events[offsets[i]:offsets[i + 1]].astype(np.int64)

    event_counts += np.bincount(seq, minlength=M)
    start_counts[seq[0]] += 1

    pairs = seq[:-1] * M + seq[1:]

    transition_counts += np.bincount(
        pairs,
        minlength=M * M
    ).reshape(M, M)

# Laplace smoothing
event_prob = (
    (event_counts + SMOOTHING) /
    (event_counts.sum() + SMOOTHING * M)
)

start_prob = (
    (start_counts + SMOOTHING) /
    (start_counts.sum() + SMOOTHING * M)
)

transition_prob = (
    (transition_counts + SMOOTHING) /
    (
        transition_counts.sum(axis=1, keepdims=True)
        + SMOOTHING * M
    )
)

print("Baseline training time:", round(time.time() - start_time, 2))
print("Probability checks:",
      np.isclose(event_prob.sum(), 1),
      np.isclose(start_prob.sum(), 1),
      np.allclose(transition_prob.sum(axis=1), 1))

# --------------------------------------------------
# 2. Implement baseline anomaly scores
# --------------------------------------------------

log_event_prob = np.log(event_prob)
log_start_prob = np.log(start_prob)
log_transition_prob = np.log(transition_prob)

def baseline_scores(indices, method):
    scores = np.empty(len(indices), dtype=np.float64)

    for j, i in enumerate(indices):
        seq = events[offsets[i]:offsets[i + 1]].astype(np.int64)

        if method == "frequency":
            log_likelihood = log_event_prob[seq].sum()

        elif method == "markov":
            log_likelihood = log_start_prob[seq[0]]

            if len(seq) > 1:
                log_likelihood += log_transition_prob[
                    seq[:-1], seq[1:]
                ].sum()

        else:
            raise ValueError("Unknown baseline")

        scores[j] = -log_likelihood / len(seq)

    return scores

# --------------------------------------------------
# 3. Validation threshold and fixed test evaluation
# --------------------------------------------------

baseline_results = []
baseline_score_outputs = {}

for method in ["frequency", "markov"]:

    print(f"\n=== {method.upper()} BASELINE ===")

    val_baseline = baseline_scores(val_indices, method)

    p, r, thresholds = precision_recall_curve(
        y_val, val_baseline
    )

    f1_curve = (
        2 * p[:-1] * r[:-1] /
        (p[:-1] + r[:-1] + 1e-12)
    )

    threshold = float(thresholds[np.argmax(f1_curve)])

    test_baseline = baseline_scores(test_indices, method)
    predictions = (test_baseline >= threshold).astype(int)

    metrics = calculate_metrics(
        y_test,
        predictions,
        test_baseline
    )

    result = {
        "model": method,
        "threshold": threshold,
        **metrics
    }

    baseline_results.append(result)

    baseline_score_outputs[f"{method}_validation"] = val_baseline
    baseline_score_outputs[f"{method}_test"] = test_baseline

    print("Threshold:", threshold)
    print("Precision:", metrics["precision"])
    print("Recall:", metrics["recall"])
    print("F1:", metrics["f1"])
    print("False-alarm rate:", metrics["false_alarm_rate"])
    print("Average precision:", metrics["average_precision"])

# --------------------------------------------------
# 4. Compare against final HMM
# --------------------------------------------------

hmm_result = {
    "model": "HMM_3state",
    "threshold": final_threshold,
    **test_metrics
}

comparison = pd.DataFrame(
    baseline_results + [hmm_result]
)

print("\n=== FINAL MODEL COMPARISON ===")

print(
    comparison[
        [
            "model",
            "precision",
            "recall",
            "f1",
            "false_alarm_rate",
            "average_precision"
        ]
    ].to_string(index=False)
)

# --------------------------------------------------
# 5. Save artifacts
# --------------------------------------------------

comparison.to_csv(
    output_dir / "final_model_comparison.csv",
    index=False
)

np.savez_compressed(
    output_dir / "baseline_models.npz",
    event_prob=event_prob,
    start_prob=start_prob,
    transition_prob=transition_prob
)

np.savez_compressed(
    output_dir / "baseline_scores.npz",
    **baseline_score_outputs
)

print("\nBaseline artifacts saved.")

Baseline training time: 4.85
Probability checks: True True True

=== FREQUENCY BASELINE ===
Threshold: 2.4509147402999014
Precision: 0.808695652173913
Recall: 1.0
F1: 0.8942307692307693
False-alarm rate: 0.0031003382187147687
Average precision: 0.8874918027775548

=== MARKOV BASELINE ===
Threshold: 2.181716543009615
Precision: 0.7980636237897649
Recall: 0.5170250896057348
F1: 0.6275149537792278
False-alarm rate: 0.0017145809845922585
Average precision: 0.6453801473668126

=== FINAL MODEL COMPARISON ===
     model  precision   recall       f1  false_alarm_rate  average_precision
 frequency   0.808696 1.000000 0.894231          0.003100           0.887492
    markov   0.798064 0.517025 0.627515          0.001715           0.645380
HMM_3state   0.695449 0.534050 0.604156          0.003065           0.692690

Baseline artifacts saved.


In [26]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import json
import shutil
import importlib.metadata
from pathlib import Path

from sklearn.metrics import (
    ConfusionMatrixDisplay,
    precision_recall_curve
)

output_dir = Path("/content/logsentinel_artifacts")
fig_dir = output_dir / "figures"
fig_dir.mkdir(exist_ok=True)

# --------------------------------------------------
# Figure 1: HMM confusion matrix
# --------------------------------------------------

fig, ax = plt.subplots(figsize=(6, 5))

ConfusionMatrixDisplay.from_predictions(
    y_test,
    test_predictions,
    labels=[0, 1],
    display_labels=["Normal", "Anomaly"],
    cmap="Blues",
    values_format="d",
    ax=ax
)

ax.set_title("HMM — Test Confusion Matrix")
fig.tight_layout()
fig.savefig(fig_dir / "hmm_confusion_matrix.png", dpi=200)
plt.close(fig)

# --------------------------------------------------
# Figure 2: Precision-recall comparison
# --------------------------------------------------

fig, ax = plt.subplots(figsize=(8, 5))

score_sets = {
    "HMM": test_scores,
    "Frequency": baseline_score_outputs["frequency_test"],
    "Markov": baseline_score_outputs["markov_test"]
}

for name, scores in score_sets.items():
    p, r, _ = precision_recall_curve(y_test, scores)
    ax.plot(r, p, label=name)

ax.set_xlabel("Recall")
ax.set_ylabel("Precision")
ax.set_title("Test Precision–Recall Curves")
ax.legend()
ax.grid(alpha=0.2)

fig.tight_layout()
fig.savefig(fig_dir / "precision_recall_comparison.png", dpi=200)
plt.close(fig)

# --------------------------------------------------
# Figure 3: Model F1 comparison
# --------------------------------------------------

fig, ax = plt.subplots(figsize=(7, 5))

comparison.plot.bar(
    x="model",
    y="f1",
    ax=ax,
    legend=False,
    color=["#16A085", "#2980B9", "#E67E22"]
)

ax.set_ylim(0, 1)
ax.set_ylabel("Test F1-score")
ax.set_xlabel("")
ax.set_title("Model Performance Comparison")
ax.tick_params(axis="x", rotation=0)

fig.tight_layout()
fig.savefig(fig_dir / "model_f1_comparison.png", dpi=200)
plt.close(fig)

# --------------------------------------------------
# Figure 4: HMM anomaly-score distributions
# --------------------------------------------------

fig, ax = plt.subplots(figsize=(8, 5))

ax.hist(
    test_scores[y_test == 0],
    bins=60,
    alpha=0.6,
    density=True,
    label="Normal"
)

ax.hist(
    test_scores[y_test == 1],
    bins=60,
    alpha=0.6,
    density=True,
    label="Anomaly"
)

ax.axvline(
    final_threshold,
    color="red",
    linestyle="--",
    label="Validation-selected threshold"
)

ax.set_xlabel("Mean Negative Log-Likelihood")
ax.set_ylabel("Density")
ax.set_title("HMM Test Anomaly-Score Distribution")
ax.legend()

fig.tight_layout()
fig.savefig(fig_dir / "hmm_score_distribution.png", dpi=200)
plt.close(fig)

# --------------------------------------------------
# Save runtime dependency versions
# --------------------------------------------------

packages = [
    "numpy", "pandas", "scipy",
    "scikit-learn", "hmmlearn",
    "matplotlib", "joblib"
]

versions = {}

for package in packages:
    versions[package] = importlib.metadata.version(package)

with open(output_dir / "dependency_versions.json", "w") as f:
    json.dump(versions, f, indent=2)

# --------------------------------------------------
# Save compact project summary
# --------------------------------------------------

project_summary = {
    "dataset": "HDFS_v1",
    "raw_log_count": 11175629,
    "labeled_blocks": 575061,
    "retained_blocks": 493860,
    "known_event_templates": 52,
    "selected_hmm_states": 3,
    "custom_forward_verified": True,
    "custom_forward_max_difference": 1.4210854715202004e-14,
    "selected_threshold": final_threshold,
    "test_metrics": test_metrics,
    "baseline_results": baseline_results,
    "split_manifest_sha256":
        "5a7be4f2e0ce4464a511295a139e3a0bff3e22271eb982fd3a86c89bd8113a2d"
}

with open(output_dir / "project_summary.json", "w") as f:
    json.dump(project_summary, f, indent=2)

# --------------------------------------------------
# Create export archive
# --------------------------------------------------

archive = shutil.make_archive(
    "/content/LogSentinel_Colab_Artifacts",
    "zip",
    root_dir="/content",
    base_dir="logsentinel_artifacts"
)

print("=== FINAL EXPORT ===")
print("Archive:", archive)
print("Archive size:", round(Path(archive).stat().st_size / 1024**2, 2), "MB")

print("\nGenerated figures:")
for file in sorted(fig_dir.glob("*.png")):
    print("-", file.name)

print("\nArtifact files:")
for file in sorted(output_dir.iterdir()):
    print("-", file.name)

print("\nCOLAB ARTIFACT PACKAGING COMPLETE")

=== FINAL EXPORT ===
Archive: /content/LogSentinel_Colab_Artifacts.zip
Archive size: 25.33 MB

Generated figures:
- hmm_confusion_matrix.png
- hmm_score_distribution.png
- model_f1_comparison.png
- precision_recall_comparison.png

Artifact files:
- baseline_models.npz
- baseline_scores.npz
- dependency_versions.json
- event_vocabulary.json
- figures
- final_model_comparison.csv
- hdfs_event_sequences.npz
- hdfs_split_manifest.csv
- hmm_final_3state.joblib
- hmm_final_evaluation.json
- hmm_final_evaluation_scores.npz
- hmm_test_predictions.csv
- hmm_training_history.npy
- hmm_training_report.json
- hmm_validation_comparison.csv
- hmm_validation_scores.npz
- project_summary.json
- retained_block_metadata.csv
- split_config.json

COLAB ARTIFACT PACKAGING COMPLETE
